In [1]:
import pandas as pd
import numpy as np
from itertools import product

# benchmarks_df.py

np.random.seed(42)

# Define dimensions
tenors = ["1M", "3M", "6M", "1Y", "2Y", "5Y", "10Y"]
tenor_years = {"1M": 1/12, "3M": 0.25, "6M": 0.5, "1Y": 1.0, "2Y": 2.0, "5Y": 5.0, "10Y": 10.0}

credit_ratings = ["AAA", "AA", "A", "BBB", "BB", "B"]
rating_spreads = {"AAA": 0.0000, "AA": 0.0008, "A": 0.0025, "BBB": 0.0075, "BB": 0.0250, "B": 0.0600}

regions = ["US", "EU", "APAC", "LATAM"]
region_basis = {"US": 0.0000, "EU": -0.0010, "APAC": -0.0005, "LATAM": 0.0100}

# Base yield curve as a simple function of tenor (in years)
def base_yield(years):
    # small upward sloping curve: intercept + slope * sqrt(years)
    return 0.005 + 0.01 * np.sqrt(years)

rows = []
for tenor, rating, region in product(tenors, credit_ratings, regions):
    yrs = tenor_years[tenor]
    base = base_yield(yrs)
    spread = rating_spreads[rating]
    region_adj = region_basis[region]
    noise = np.random.normal(loc=0.0, scale=0.0005)  # small market noise
    yield_decimal = base + spread + region_adj + noise
    rows.append({
        "tenor": tenor,
        "tenor_years": yrs,
        "credit_rating": rating,
        "region": region,
        "yield": round(yield_decimal, 6),        # decimal form (e.g., 0.015)
        "yield_bp": int(round(yield_decimal * 10000))  # basis points as integer
    })

benchmarks_df = pd.DataFrame(rows).sort_values(["region", "credit_rating", "tenor_years"]).reset_index(drop=True)

# Example: pivot to view yields by tenor across ratings for a region
# pivot_example = benchmarks_df.pivot_table(index="tenor", columns="credit_rating", values="yield")

benchmarks_df.head()

,tenor,tenor_years,credit_rating,region,yield,yield_bp
0,1M,0.083333,A,APAC,0.009655,97
1,3M,0.250000,A,APAC,0.012411,124
2,6M,0.500000,A,APAC,0.014237,142
3,1Y,1.000000,A,APAC,0.017739,177
4,2Y,2.000000,A,APAC,0.022085,221


In [5]:
# Manual OLS + Breusch-Pagan implementation to avoid dependency/import issues
# Uses existing variables: benchmarks_df, tenor_years, etc.

import scipy.stats as stats

df = benchmarks_df.copy()

# Build design matrix: numeric tenor_years + dummies for credit_rating and region
X_df = pd.get_dummies(df[['tenor_years', 'credit_rating', 'region']], drop_first=True)
y = df['yield'].values

# add constant column manually (do not rely on statsmodels import)
n = len(y)
X_mat = np.column_stack((np.ones(n), X_df.values))  # shape (n, k)
k = X_mat.shape[1]

# OLS via numpy
beta = np.linalg.lstsq(X_mat, y, rcond=None)[0]
y_hat = X_mat @ beta
resid = y - y_hat

# Auxiliary regression: squared residuals on the original exog (including constant)
aux_y = resid**2
# Fit auxiliary regression
beta_aux = np.linalg.lstsq(X_mat, aux_y, rcond=None)[0]
aux_hat = X_mat @ beta_aux

# Compute R-squared for auxiliary regression
ssr = np.sum((aux_hat - aux_y.mean())**2)
sst = np.sum((aux_y - aux_y.mean())**2)
R2 = ssr / sst if sst > 0 else 0.0

# LM statistic and p-value (chi-square with df = k-1)
lm_stat = n * R2
lm_pvalue = 1.0 - stats.chi2.cdf(lm_stat, df=k-1)

# F-statistic (as in statsmodels implementation) and p-value
# guard against division by zero
if (n - k) > 0 and (1 - R2) > 0:
    f_stat = (R2 / (k - 1)) / ((1 - R2) / (n - k))
    f_pvalue = 1.0 - stats.f.cdf(f_stat, k - 1, n - k)
else:
    f_stat = np.nan
    f_pvalue = np.nan

results = {
    'lm_stat': lm_stat,
    'lm_pvalue': lm_pvalue,
    'f_stat': f_stat,
    'f_pvalue': f_pvalue
}

print("Breusch-Pagan test results:", results)

alpha = 0.05
if results['lm_pvalue'] < alpha:
    print(f"Result: reject homoscedasticity (p={results['lm_pvalue']:.4f}) — evidence of heteroscedasticity.")
else:
    print(f"Result: cannot reject homoscedasticity (p={results['lm_pvalue']:.4f}).")

UFuncTypeError: Cannot cast ufunc 'lstsq' input 0 from dtype('O') to dtype('float64') with casting rule 'same_kind'

In [3]:
results

NameError: name 'results' is not defined